# Phase 3 — Hybrid ML + DL Intrusion Detection

## 1. Motivation
Phase 1 produced two classical baselines (Z-Score, Isolation Forest). Phase 2 produced three deep one-class detectors (One-Class SVM, Autoencoder with skips, β-VAE). The Phase-2 evaluation surfaced a concrete weakness:

| Model (Phase 2) | Accuracy | Precision | Recall | F1 | ROC-AUC |
|---|---|---|---|---|---|
| Autoencoder (Skip) | 0.582 | 0.576 | **1.000** | 0.731 | 0.932 |
| VAE (β = 0.5)      | 0.582 | 0.577 | **1.000** | 0.732 | 0.941 |

Recall is saturated at 1.0 — every attack is caught — but precision collapses to ≈0.58. The reconstruction-error threshold is too permissive: every uncommon record is flagged, including benign ones. The deep models contain useful information (AUC > 0.93) but the *decision rule* is poorly calibrated.

Phase 3 attacks this in two complementary ways by combining ML and DL components into hybrid architectures.

## 2. Hybrid Architectures

### 2.1 Model A — Deep Isolation Forest (DIF)
**Pipeline:** raw record → **Autoencoder encoder** → 32-D latent vector `z` → **Isolation Forest** → anomaly score.

*Why this is a real hybrid, not glue:*
- The autoencoder solves Isolation Forest's curse-of-dimensionality on heterogeneous tabular features. IF on the raw 43-D NSL-KDD input wastes splits on highly correlated columns.
- Isolation Forest replaces the autoencoder's brittle MSE threshold with a principled isolation-based score — path length in random binary trees — that is scale-free and well-calibrated even when the latent distribution is non-Gaussian.
- Each component fixes the other's weakness — strong coupling, not stacking.

**Reference (paper of inspiration):** Xu, Y., Pang, G., Wang, Y., & Wang, Y. (2023). *Deep Isolation Forest for Anomaly Detection.* IEEE TKDE.

### 2.2 Model C — Cascaded AE + XGBoost (CAX)
**Pipeline:** raw record → **Autoencoder** → produces `z`, `x_hat`, residuals `|x - x_hat|`. The classifier feature vector is the concatenation `[x ‖ z ‖ |x - x_hat|]`. **XGBoost** on this enriched vector produces a 5-class label `{Normal, DoS, Probe, R2L, U2R}`.

Optionally, AE acts as a **Stage-1 gate**: records with reconstruction error below the validation-tuned threshold short-circuit to Normal without invoking XGBoost.

*Why this is a real hybrid, not glue:*
- The DL component contributes **representation learning** (latent `z`) and a **per-feature error signal** (residual vector) — XGBoost cannot derive these from the raw features.
- The ML component contributes **labeled supervision** and **interpretability** — feature importances tell us *which* AE-derived dimensions actually drive the classification, and the per-class report tells us *which* attack family the system fails on.
- The AE's main Phase-2 failure mode (over-flagging) is converted into useful information: the residual vector is exactly the signal that a downstream supervised model can exploit.

**Reference (paper of inspiration):** Aygun, R. C., & Yavuz, A. G. (2017). *Network Anomaly Detection with Stochastically Improved Autoencoder Based Models.* IEEE CSCloud.

## 3. Architecture Diagram (textual)
Detailed visual versions live in `figures/diagram_model_A.png` and `figures/diagram_model_C.png`, generated in notebook 06.

```
Model A — Deep Isolation Forest
─────────────────────────────────────────────────────────────
  x ∈ ℝ^43  ──►  AE.encoder  ──►  z ∈ ℝ^32  ──►  IsolationForest  ──►  score ∈ ℝ
                  (DL)                            (ML)
  Trained on:    train-normal MSE              latent z of train-normal
  Threshold:     tuned on val_mixed by max-F1

Model C — Cascaded AE + XGBoost
─────────────────────────────────────────────────────────────
                       ┌────────────► z      (latent, ℝ^32)
  x ∈ ℝ^43 ─► AE ──────┤
      │                └────────────► x_hat  (reconstruction)
      │                                ↓
      │                          residual = |x - x_hat|  (ℝ^43)
      │
      ▼  (optional Stage-1 gate: skip XGBoost if MSE < τ)
  concat(x, z, residual)  ∈ ℝ^118  ──►  XGBoost  ──►  ŷ ∈ {Normal, DoS, Probe, R2L, U2R}
       (DL-derived feats)                  (ML)
```

## 4. Notebook Map
| # | Notebook | Purpose |
|---|---|---|
| 02 | `02_data_preparation.ipynb` | NSL-KDD load, leakage-safe splits, persistence to `results/processed_data.npz` |
| 03 | `03_deep_isolation_forest.ipynb` | Model A — train AE, fit IF on latent, evaluate |
| 04 | `04_cascaded_ae_xgboost.ipynb`  | Model C — build hybrid features, train XGBoost, evaluate |
| 05 | `05_ablation_study.ipynb`       | Per-component ablations (ML-only, DL-only, partial hybrids, full hybrid) |
| 06 | `06_final_comparison.ipynb`     | Phase 1 vs Phase 2 vs Phase 3 consolidation + figures + report |

## 5. Rubric Mapping
| Rubric component | Evidence |
|---|---|
| **Hybrid Innovation** | Each model couples DL and ML so that one fixes a specific weakness of the other (Section 2). |
| **Ablation Studies** | Notebook 05 produces a single table comparing AE-only, IF-on-raw, Deep-IF, XGB-on-raw, XGB-on-raw+z, full Model C, and cascade — on the same test set. |
| **Architecture Diagram** | Notebook 06 renders publication-style diagrams with tensor shapes and explicit fusion mechanism. |
| **Reproducibility** | Self-contained `Phase 3/`, `requirements.txt`, no hardcoded paths, deterministic seeds, instructions in `README.md`, optional `Dockerfile`. |
| **Extra Mile** | Streamlit demo (`app/streamlit_app.py`), Dockerfile, SHAP-on-residuals analysis. |